# Model Optimization: Pruning

In this notebook, we'll apply pruning techniques to our models using distributed processing. Instead of running the pruning on our notebook instance, we'll launch separate SageMaker Processing jobs to perform the pruning on more powerful instances.

## What is Pruning?

Pruning is a technique that removes unnecessary weights from a neural network, effectively making the model more sparse. Research has shown that many neural networks are overparameterized, and a significant percentage of weights can be removed without substantial impact on accuracy.

### Types of Pruning

#### Unstructured Pruning
- **Description**: Removes individual weights based on importance criteria (typically magnitude)
- **Advantages**: Higher theoretical compression rates, more fine-grained control
- **Disadvantages**: Requires specialized hardware/software for speed benefits
- **Example**: Setting the smallest 30% of weights to zero based on their absolute values

#### Structured Pruning
- **Description**: Removes entire structures like neurons, channels, or attention heads
- **Advantages**: Immediate speed benefits on standard hardware, actual size reduction
- **Disadvantages**: Generally higher accuracy impact than unstructured pruning
- **Example**: Removing entire neurons or attention heads based on their importance

In this notebook, we'll focus on structured pruning to achieve actual size reduction and inference speedup.

### Benefits of Pruning:
- **Reduced Model Size**: Fewer parameters means smaller models
- **Faster Inference**: Fewer computations lead to faster inference
- **Lower Memory Requirements**: Sparse models require less memory
- **Reduced Overfitting**: Removing redundant weights can improve generalization

### Distributed Processing Approach
This notebook uses SageMaker Processing jobs to perform pruning on separate, more powerful instances. This approach allows us to:
1. Use a small, cost-effective instance for our notebook
2. Launch larger instances only when needed for resource-intensive tasks
3. Process multiple models in parallel

## 1. Import Dependencies

In [ ]:
import json
import time
import pandas as pd
import boto3
import sagemaker
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.pytorch.processing import PyTorchProcessor
import time
from IPython.display import clear_output

## 2. Load Workshop Settings

Load the workshop settings that were configured in the first notebook.

In [ ]:
# Load stored variables
%store -r S3_BUCKET
%store -r AWS_REGION
%store -r SAGEMAKER_ROLE_ARN
%store -r OPTIMIZATION_INSTANCE_TYPE

# Check if variables were successfully retrieved
if 'S3_BUCKET' in locals() and S3_BUCKET != "YOUR_BUCKET_NAME_HERE":
    print("Workshop settings loaded successfully:")
    print(f"S3 Bucket: {S3_BUCKET}")
    print(f"AWS Region: {AWS_REGION}")
    print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")
    print(f"Optimization Instance Type: {OPTIMIZATION_INSTANCE_TYPE}")
else:
    print("⚠️ Workshop settings not found or not configured.")
    print("Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.")
    
    # Set default values that user should update
    S3_BUCKET = "YOUR_BUCKET_NAME_HERE"  # Update this value
    AWS_REGION = "YOUR_REGION_HERE"      # Update this value
    SAGEMAKER_ROLE_ARN = "YOUR_ROLE_ARN_HERE"  # Update this value
    OPTIMIZATION_INSTANCE_TYPE = "ml.c5.xlarge"  # Default optimization instance type
    
    # Store the updated values
    %store S3_BUCKET
    %store AWS_REGION
    %store SAGEMAKER_ROLE_ARN
    %store OPTIMIZATION_INSTANCE_TYPE

## 3. Load Model Information

In [ ]:
# Try to load model information from file
try:
    with open('model_info.json', 'r') as f:
        model_info = json.load(f)
    print(f"Loaded information for {len(model_info)} models")
except FileNotFoundError:
    print("model_info.json not found. Using default model information.")
    model_info = {
        "sentiment-analysis": {
            "model_name": "distilbert-base-uncased-finetuned-sst-2-english",
            "task": "text-classification",
            "hub_model_id": "distilbert-base-uncased-finetuned-sst-2-english",
            "s3_uri": f"s3://{S3_BUCKET}/models/distilbert-base-uncased-finetuned-sst-2-english"
        }
    }

## 4. Define Sample Inputs for Each Task

In [ ]:
# Define sample inputs for each task
sample_inputs = {
    "sentiment-analysis": "I really enjoyed this movie. The acting was superb and the plot was engaging.",
    "ner": "Jeff Bezos founded Amazon in 1994 and the company is headquartered in Seattle, Washington.",
    "question-answering": {
        "question": "What is machine learning?",
        "context": "Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data."
    },
    "masked-lm": "The [MASK] is a large language model trained by OpenAI."
}

## 5. Create Pruning Script

In this section, we'll create a Python script that performs the actual pruning. This script will be executed on the SageMaker Processing instances.

### What the Script Does:
1. **Loads the model and tokenizer** from Hugging Face
2. **Prepares sample inputs** for inference
3. **Applies pruning** using the specified method and amount
4. **Measures performance metrics** like model size and inference time
5. **Saves the pruned model** and metrics to the output directory

### Pruning Methods:
- **Structured Pruning**: Removes entire structures (like neurons) based on their importance
- **Advanced Pruning**: Uses specialized libraries for optimized pruning with actual size reduction
- **L1 Unstructured Pruning**: Sets individual weights to zero based on their L1 norm

The pruning amount parameter controls what percentage of weights to remove. For example, a value of 0.3 means 30% of weights will be pruned.

## 6. Launch Distributed Pruning Jobs

Now we'll set up and launch the SageMaker Processing jobs to perform pruning. Each model will be processed in a separate job, allowing for parallel processing.

### Pruning Process:
1. **Create a PyTorch processor** with the appropriate instance type and configuration
2. **For each model**:
   - Save and upload model information to S3
   - Define inputs (pruning script and model info) and outputs
   - Launch a processing job with the appropriate arguments
   - Store the job information for monitoring

We're using structured pruning with a pruning amount of 0.3 (30% of weights will be removed). This method removes entire neurons/filters, which actually reduces the model size unlike unstructured pruning.

### Parallel Processing
To speed up the process, we'll launch all jobs in parallel rather than waiting for each job to complete before starting the next one.

In [ ]:
# Define the instance type to use for pruning
instance_type = OPTIMIZATION_INSTANCE_TYPE
print(f"Using instance type: {instance_type} for optimization jobs")

# Create a SageMaker session
sagemaker_session = sagemaker.Session()

# Create a PyTorch processor
processor = PyTorchProcessor(
    framework_version="2.0.0",
    py_version="py310",
    role=SAGEMAKER_ROLE_ARN,
    instance_type=instance_type,
    instance_count=1,
    base_job_name="model-pruning",
    sagemaker_session=sagemaker_session
)

In [ ]:
# Launch pruning jobs for all models in parallel
job_names = []  # List to store all job names
job_output_paths = {}
s3_client = boto3.client('s3')

# First, prepare all the job configurations
job_configs = {}
print("Preparing pruning jobs for all models...")

for model_key in model_info.keys():
    # Save model info to a temporary file
    with open(f'temp_{model_key}_info.json', 'w') as f:
        json.dump({model_key: model_info[model_key]}, f)
    
    # Upload to S3
    s3_client.upload_file(
        f'temp_{model_key}_info.json', 
        S3_BUCKET, 
        f'optimization/inputs/{model_key}/model_info.json'
    )
    
    # Define the output path
    output_path = f's3://{S3_BUCKET}/optimization/outputs/{model_key}-pruned'  # Using hyphens consistently
    job_output_paths[model_key] = output_path
    
    # Define inputs and outputs
    inputs = [
        ProcessingInput(
            source=f's3://{S3_BUCKET}/optimization/inputs/{model_key}/model_info.json',
            destination='/opt/ml/processing/input/data'
        )
    ]
    
    outputs = [
        ProcessingOutput(
            output_name='pruned-model',  # Using hyphens consistently
            source='/opt/ml/processing/output',
            destination=output_path
        )
    ]
    
    # Store the job configuration
    job_configs[model_key] = {
        'inputs': inputs,
        'outputs': outputs,
        'arguments': [
            '--model-info-path', '/opt/ml/processing/input/data/model_info.json',
            '--output-dir', '/opt/ml/processing/output',
            '--pruning-method', 'structured',  # Using structured pruning for actual size reduction
            '--pruning-amount', '0.3'
        ]
    }
    print(f"Prepared job configuration for {model_key}")

# Now launch all jobs in parallel
print("\nLaunching all pruning jobs in parallel...")
for model_key, config in job_configs.items():
    try:
        # Create a unique job name with timestamp to avoid conflicts
        timestamp = int(time.time())
        job_name = f"pruning-{model_key}-{timestamp}"
        
        # Run the processing job with the unique name
        processor.run(
            code='pruning_script.py',
            source_dir='pruning_scripts',
            inputs=config['inputs'],
            outputs=config['outputs'],
            arguments=config['arguments'],
            wait=False,  # Don't wait for the job to complete before continuing
            job_name=job_name  # Explicitly set the job name
        )
        
        # Store the job name for tracking
        job_names.append(job_name)
        print(f"Launched job for {model_key}: {job_name}")
    except Exception as e:
        print(f"Error launching job for {model_key}: {e}")

print("\nAll jobs launched. You can monitor their progress in the SageMaker console.")

In [ ]:
# Monitor job status
from sagemaker.processing import ProcessingJob

# Function to check if all jobs are complete
def are_all_jobs_complete(job_names, sagemaker_session):
    all_complete = True
    job_statuses = {}
    
    # Create a SageMaker client for API calls
    sagemaker_client = boto3.client('sagemaker')
    
    for job_name in job_names:
        try:
            # Use the SageMaker client to describe the processing job
            response = sagemaker_client.describe_processing_job(
                ProcessingJobName=job_name
            )
            status = response['ProcessingJobStatus']
            job_statuses[job_name] = status
            
            if status in ['InProgress', 'Stopping']:
                all_complete = False
        except Exception as e:
            job_statuses[job_name] = f"Error: {str(e)}"
            # Consider jobs with errors as complete to avoid infinite loops
            
    return all_complete, job_statuses

# Poll for job completion
print("Waiting for all jobs to complete...")
while True:
    all_complete, job_statuses = are_all_jobs_complete(job_names, sagemaker_session)
    
    # Clear previous output
    clear_output(wait=True)
    
    # Print current status
    print("Current job statuses:")
    for job_name, status in job_statuses.items():
        print(f"Job {job_name}: {status}")
    
    if all_complete:
        print("All jobs completed!")
        break
    
    print("Waiting for jobs to complete... Will check again in 60 seconds.")
    time.sleep(60)  # Check every minute

print("\nAll jobs have completed or failed.")

## 7. Analyze Pruned Models

Now that the pruning jobs are complete, we'll analyze the pruned models by comparing them to the original models. We'll measure:

1. **Size Reduction**: How much smaller the pruned model is
2. **Parameter Reduction**: How many parameters were removed
3. **Sparsity**: The percentage of zero weights in the model
4. **Inference Time Improvement**: How much faster the pruned model is
5. **Memory Usage Reduction**: How much less memory the pruned model uses

First, let's define the functions we'll use to collect metrics:

In [ ]:
# Define functions for metrics collectiondef get_model_size(model):    """Calculate model size in MB."""    param_size = 0    for param in model.parameters():        param_size += param.nelement() * param.element_size()    buffer_size = 0    for buffer in model.buffers():        buffer_size += buffer.nelement() * buffer.element_size()        size_mb = (param_size + buffer_size) / 1024**2    return size_mbdef get_num_parameters(model):    """Calculate number of parameters in the model."""    return sum(p.numel() for p in model.parameters())def count_non_zero_params(model):    """Count non-zero parameters in the model."""    non_zero = 0    total = 0    for param in model.parameters():        if param.dim() > 1:  # Only count weights, not biases            non_zero += torch.count_nonzero(param).item()            total += param.numel()    return non_zero, totaldef measure_inference_time(model, inputs, num_runs=10):    """Measure average inference time over multiple runs."""    # Warm-up run    with torch.no_grad():        model(**inputs)        # Measure inference time    start_event = torch.cuda.Event(enable_timing=True) if torch.cuda.is_available() else None    end_event = torch.cuda.Event(enable_timing=True) if torch.cuda.is_available() else None        inference_times = []    for _ in range(num_runs):        if torch.cuda.is_available():            start_event.record()            with torch.no_grad():                model(**inputs)            end_event.record()            torch.cuda.synchronize()            inference_times.append(start_event.elapsed_time(end_event))        else:            start_time = time.time()            with torch.no_grad():                model(**inputs)            end_time = time.time()            inference_times.append((end_time - start_time) * 1000)  # Convert to ms        return sum(inference_times) / len(inference_times)def measure_memory_usage(model, inputs):    """Measure peak memory usage during inference."""    if torch.cuda.is_available():        torch.cuda.reset_peak_memory_stats()        torch.cuda.empty_cache()                with torch.no_grad():            model(**inputs)                memory_usage = torch.cuda.max_memory_allocated() / 1024**2  # Convert to MB    else:        # For CPU, use a rough estimate based on model size        memory_usage = get_model_size(model) * 2  # Rough estimate        return memory_usagedef prepare_sample_inputs(model_name, task, tokenizer, device):    """Prepare sample inputs for the model based on its task."""    if task == "sequence-classification" or task == "text-classification":        text = "I really enjoyed this movie. The acting was superb and the plot was engaging."        inputs = tokenizer(text, return_tensors="pt")    elif task == "token-classification":        text = "Jeff Bezos founded Amazon in 1994 and the company is headquartered in Seattle, Washington."        inputs = tokenizer(text, return_tensors="pt")    elif task == "question-answering":        question = "What is machine learning?"        context = "Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data."        inputs = tokenizer(question, context, return_tensors="pt")    elif task == "masked-lm" or task == "fill-mask":        text = "The [MASK] is a large language model trained by OpenAI."        inputs = tokenizer(text, return_tensors="pt")    else:        raise ValueError(f"Unsupported task: {task}")        # Move inputs to the appropriate device    return {k: v.to(device) for k, v in inputs.items()}

In [ ]:
# Collect metrics for all pruned modelsall_metrics = {}for model_key, job_info in job_configs.items():    print(f"\nAnalyzing pruned model: {model_key}")        # Get model info    model_info = model_info_dict[model_key]    model_name = model_info["model_name"]    task = model_info["task"]        # Set device    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")    print(f"Using device: {device}")        # Load original model    print(f"Loading original model: {model_name}")    tokenizer = AutoTokenizer.from_pretrained(model_name)        if task == "sequence-classification" or task == "text-classification":        original_model = AutoModelForSequenceClassification.from_pretrained(model_name)    elif task == "token-classification":        original_model = AutoModelForTokenClassification.from_pretrained(model_name)    elif task == "question-answering":        original_model = AutoModelForQuestionAnswering.from_pretrained(model_name)    elif task == "masked-lm" or task == "fill-mask":        original_model = AutoModelForMaskedLM.from_pretrained(model_name)    else:        raise ValueError(f"Unsupported task: {task}")        original_model = original_model.to(device)    original_model.eval()        # Prepare sample inputs    inputs = prepare_sample_inputs(model_name, task, tokenizer, device)        # Measure baseline metrics    baseline_size = get_model_size(original_model)    baseline_params = get_num_parameters(original_model)    baseline_non_zero, baseline_total = count_non_zero_params(original_model)    baseline_inference_time = measure_inference_time(original_model, inputs)    baseline_memory_usage = measure_memory_usage(original_model, inputs)        print(f"Original model size: {baseline_size:.2f} MB")    print(f"Original parameters: {baseline_params:,}")    print(f"Original non-zero weights: {baseline_non_zero:,}/{baseline_total:,} ({baseline_non_zero/baseline_total*100:.2f}%)")    print(f"Original inference time: {baseline_inference_time:.2f} ms")    print(f"Original memory usage: {baseline_memory_usage:.2f} MB")        # Load pruned model    pruned_model_path = os.path.join(job_info["output_path"], f"{model_key}_pruned")    print(f"Loading pruned model from: {pruned_model_path}")        try:        if task == "sequence-classification" or task == "text-classification":            pruned_model = AutoModelForSequenceClassification.from_pretrained(pruned_model_path)        elif task == "token-classification":            pruned_model = AutoModelForTokenClassification.from_pretrained(pruned_model_path)        elif task == "question-answering":            pruned_model = AutoModelForQuestionAnswering.from_pretrained(pruned_model_path)        elif task == "masked-lm" or task == "fill-mask":            pruned_model = AutoModelForMaskedLM.from_pretrained(pruned_model_path)                pruned_model = pruned_model.to(device)        pruned_model.eval()                # Measure pruned metrics        pruned_size = get_model_size(pruned_model)        pruned_params = get_num_parameters(pruned_model)        pruned_non_zero, pruned_total = count_non_zero_params(pruned_model)        pruned_inference_time = measure_inference_time(pruned_model, inputs)        pruned_memory_usage = measure_memory_usage(pruned_model, inputs)                print(f"Pruned model size: {pruned_size:.2f} MB")        print(f"Pruned parameters: {pruned_params:,}")        print(f"Pruned non-zero weights: {pruned_non_zero:,}/{pruned_total:,} ({pruned_non_zero/pruned_total*100:.2f}%)")        print(f"Pruned inference time: {pruned_inference_time:.2f} ms")        print(f"Pruned memory usage: {pruned_memory_usage:.2f} MB")                # Calculate improvements        size_reduction = (baseline_size - pruned_size) / baseline_size * 100        param_reduction = (baseline_params - pruned_params) / baseline_params * 100        sparsity = (1 - pruned_non_zero / pruned_total) * 100        time_improvement = (baseline_inference_time - pruned_inference_time) / baseline_inference_time * 100        memory_reduction = (baseline_memory_usage - pruned_memory_usage) / baseline_memory_usage * 100                print(f"Size reduction: {size_reduction:.2f}%")        print(f"Parameter reduction: {param_reduction:.2f}%")        print(f"Model sparsity: {sparsity:.2f}%")        print(f"Inference time improvement: {time_improvement:.2f}%")        print(f"Memory usage reduction: {memory_reduction:.2f}%")                # Save metrics        all_metrics[model_key] = {            "model_name": model_name,            "task": task,            "pruning_method": "structured",            "pruning_amount": 0.3,            "model_size": round(pruned_size, 2),            "inference_time": round(pruned_inference_time, 2),            "memory_usage": round(pruned_memory_usage, 2),            "size_reduction": round(size_reduction, 2),            "time_improvement": round(time_improvement, 2),            "memory_reduction": round(memory_reduction, 2),            "parameter_reduction": round(param_reduction, 2),            "sparsity": round(sparsity, 2),            "non_zero_weights": pruned_non_zero,            "total_weights": pruned_total        }            except Exception as e:        print(f"Error analyzing pruned model {model_key}: {e}")        import traceback        traceback.print_exc()# Save all metrics to a filemetrics_path = "pruned-metrics.json"with open(metrics_path, "w") as f:    json.dump(all_metrics, f, indent=2)print(f"\nSaved pruned metrics to {metrics_path}")

In [ ]:
# Download and combine results
pruned_metrics = {}
sagemaker_client = boto3.client("sagemaker")

for model_key in model_info.keys():
    # Check if we have a job for this model
    if model_key not in job_output_paths:
        print(f"No output path found for {model_key}, skipping metrics collection")
        continue
        
    # Download metrics file
    try:
        # Use the saved output path - match the filename in pruning_script.py (line 429)
        s3_client.download_file(
            S3_BUCKET,
            f'optimization/outputs/{model_key}-pruned/pruned-metrics.json',  # Using hyphens consistently
            f'temp_{model_key}-pruned-metrics.json'  # Keep hyphenated local filename
        )
        
        # Load metrics
        with open(f'temp_{model_key}-pruned-metrics.json', 'r') as f:
            metrics = json.load(f)
        
        # Add to combined metrics
        pruned_metrics.update(metrics)
        
        print(f"Downloaded metrics for {model_key}")
    except Exception as e:
        print(f"Error downloading metrics for {model_key}: {e}")

# Save combined metrics
with open('pruned-metrics.json', 'w') as f:  # Using hyphens consistently
    json.dump(pruned_metrics, f, indent=2)

print(f"\nSaved pruned metrics for {len(pruned_metrics)} models to pruned-metrics.json")

## 8. Analyze Model Size Reduction

Now we'll analyze the size reduction achieved through pruning. This analysis helps us understand the impact of pruning on model size and memory footprint.

In [ ]:
# Create a DataFrame for comparison
comparison_data = []

# Function to get total size of objects with a prefix from S3
def get_total_size(bucket, prefix):
    total_size = 0
    paginator = s3_client.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        if 'Contents' in page:
            for obj in page['Contents']:
                total_size += obj['Size']
    return total_size

for model_key in pruned_metrics.keys():
    pruned = pruned_metrics[model_key]
    
    # Get original model size from S3
    original_model_prefix = model_info[model_key]['s3_uri'].replace(f"s3://{S3_BUCKET}/", "")
    original_size = get_total_size(S3_BUCKET, original_model_prefix)
    original_size_mb = original_size / (1024 * 1024)
    
    # Get pruned model size
    pruned_model_size_mb = pruned.get('model_size', 0)
    
    # Calculate size reduction percentage
    size_reduction = ((original_size_mb - pruned_model_size_mb) / original_size_mb) * 100 if original_size_mb > 0 else 0
    
    # Prepare data for this model
    model_data = {
        'Model': pruned['model_name'],
        'Original Size (MB)': f"{original_size_mb:.2f}",
        'Pruning Method': pruned['pruning_method'],
        'Pruning Amount': f"{pruned['pruning_amount'] * 100:.1f}%",
        'Pruned Size (MB)': f"{pruned_model_size_mb:.2f}",
        'Size Reduction (%)': f"{size_reduction:.2f}",
        'Sparsity (%)': f"{pruned.get('sparsity', 0):.2f}"
    }
    
    comparison_data.append(model_data)

# Create DataFrame
comparison_df = pd.DataFrame(comparison_data)

# Display the DataFrame
comparison_df

## 9. Next Steps

Now that we've applied pruning to our models and analyzed the size reduction, we'll explore WANDA pruning in the next notebook to create even smaller, more efficient models.

### What We've Learned:
- How to apply structured pruning to transformer models for actual size reduction
- How to use SageMaker Processing for distributed optimization tasks
- How different pruning methods affect model size and memory footprint
- How to run multiple processing jobs in parallel for faster experimentation

### What's Next - WANDA Pruning:
WANDA (Weight ANalysis for Deep leArning) pruning is an advanced technique that considers both weight magnitudes and activation statistics when deciding which weights to prune. This approach tends to preserve model accuracy better than simple magnitude-based pruning.